# Phonemization (SYNCED to `IPA_Phonemes.xlsx`)
Standalone batch phonemizer — drop-in replacement for the original `Phonemization.ipynb`.

**Key change:** `IPA_RULES` is no longer hand-typed. It is **built from `IPA_Phonemes.xlsx`** at runtime (single source of truth); if the xlsx is not found, an embedded fallback — auto-generated from the same xlsx and verified byte-identical — is used. This permanently eliminates code/spreadsheet drift.

Faithful to the xlsx: Gulf/Levantine articles use **/ɪ/**, **ل is excluded** from sun letters (moon exception), Egyptian **ء→/æ/**, and the full hamza-carrier set **إ آ ئ ؤ** is included. Bugs fixed vs the original: Latin `f`/`d` typos gone (now sourced from xlsx), `Al-`/`للـ` anchored to word start, bare-alif `/aː/` recovered.

# 1. Setup

In [ ]:
!pip install -q pandas openpyxl
import pandas as pd, os
from google.colab import drive
drive.mount('/content/drive')
print("Ready.")

# 2. IPA Rules (EXPERT-REVISED, loaded from the spreadsheet) + G2P engine

In [ ]:
import json, re
# ============================================================
# Arabic G2P — rules loaded from IPA_Phonemes.xlsx (single source of truth),
# expert-revised: lam is a sun letter; hamza = /ʔ/; bare alif = /aː/ (word-initial /ʔ/);
# short vowels via diacritize-then-G2P (DIACRITIZE_HOOK).
# ============================================================
XLSX_PATH = "/content/drive/MyDrive/IPA_Phonemes.xlsx"   # <-- set to your (revised) xlsx

SUN_LETTERS  = set("تثدذرزسشصضطظلن")     # 14, INCLUDING lam
MOON_LETTERS = set("ءأإآبجحخعغفقكمهوي")

def _clean(x): return "" if x is None else str(x).replace("\t","").strip()
def _norm(p):
    p=_clean(p)
    if p=="" or p.lower() in ("silent","no vowel"): return ""
    if p.lower()=="consonant doubling": return "doubled"
    return p

def load_ipa_rules_from_xlsx(path):
    import openpyxl
    wb=openpyxl.load_workbook(path, data_only=True); ws=wb['IPA']
    DIAL_COL={"Egyptian":2,"Khaliji Arabic":6,"Levantine Arabic":10,"Modern Standard Arabic (MSA)":14}
    letters={d:{} for d in DIAL_COL}
    for r in ws.iter_rows(values_only=True,min_row=3,max_row=ws.max_row):
        g=_clean(r[1])
        if not g: continue
        for d,ci in DIAL_COL.items():
            ph=_norm(r[ci])
            if ph!="" and g not in letters[d]: letters[d][g]={"phoneme":ph}
    wsd=wb['Diacritic']; diac={}
    for r in wsd.iter_rows(values_only=True,min_row=2,max_row=9):
        k=_clean(r[0])
        if k: diac[k]=_norm(r[1])
    short={k:diac[k] for k in ("\u064e","\u064f","\u0650") if k in diac}
    tanwin={k:diac[k] for k in ("\u064b","\u064c","\u064d") if k in diac}
    other={"\u0652":diac.get("\u0652",""),"\u0651":"doubled"}
    wsm=wb['Sun and Moon Letters']
    rows=[[_clean(c) for c in row[:8]] for row in wsm.iter_rows(values_only=True,min_row=3,max_row=7)]
    moon=next(r for r in rows if r[0]=="Moon Letters"); sun=next(r for r in rows if r[0]=="Sun Letters")
    DCOL={"Egyptian":3,"Khaliji Arabic":4,"Levantine Arabic":5,"Modern Standard Arabic (MSA)":6}
    fa=lambda s:s.split(" or ")[0].split(" +")[0].strip()
    sun_set=set(sun[1].replace(",","").replace(" ",""))   # now includes ل
    R={"dialects":{}}
    for d in DIAL_COL:
        R["dialects"][d]={"letters":letters[d],
            "vowels":{"short":dict(short),"long":{"ا":"/aː/","و":"/uː/","ي":"/iː/"}},
            "diacritics":{**tanwin,**other},
            "phonological_rules":{
                "al_prefix_moon":{"letters":moon[1].replace("هـ","ه"),"ipa":fa(moon[DCOL[d]])},
                "al_prefix_sun":{"letters":"".join(sorted(sun_set)),"ipa":fa(sun[DCOL[d]])}},
            "lil_prefix":"/lel-/" if d=="Egyptian" else ("/lal-/" if "MSA" in d else "/lil-/")}
        R["dialects"][d]["letters"].setdefault("ا",{"phoneme":"/aː/"})
    return R

try:
    IPA_RULES = load_ipa_rules_from_xlsx(XLSX_PATH)
    print(f"[IPA_RULES] loaded from xlsx: {XLSX_PATH}")
except Exception as e:
    print(f"[IPA_RULES] xlsx unavailable ({e}); using embedded fallback.")
    IPA_RULES = json.loads(r'''{"dialects": {"Egyptian": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/t/"}, "ج": {"phoneme": "/ɡ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/d/"}, "ر": {"phoneme": "/ɾ/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/zˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/ʔ/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/ or silent"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/el-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/e-/"}}, "lil_prefix": "/lel-/"}, "Khaliji Arabic": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/θ/"}, "ج": {"phoneme": "/dʒ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/ð/"}, "ر": {"phoneme": "/r/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/ðˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/g/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/ɪl-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/ɪ-/"}}, "lil_prefix": "/lil-/"}, "Levantine Arabic": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/t/"}, "ج": {"phoneme": "/ʒ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/d/"}, "ر": {"phoneme": "/r/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/zˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/ʔ/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/ɪl-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/ɪ-/"}}, "lil_prefix": "/lil-/"}, "Modern Standard Arabic (MSA)": {"letters": {"ء": {"phoneme": "/ʔ/"}, "أ": {"phoneme": "/ʔ/"}, "إ": {"phoneme": "/ʔi/"}, "آ": {"phoneme": "/ʔaː/"}, "ئ": {"phoneme": "/ʔ/"}, "ؤ": {"phoneme": "/ʔ/"}, "ب": {"phoneme": "/b/"}, "ت": {"phoneme": "/t/"}, "ث": {"phoneme": "/θ/"}, "ج": {"phoneme": "/dʒ/"}, "ح": {"phoneme": "/ħ/"}, "خ": {"phoneme": "/x/"}, "د": {"phoneme": "/d/"}, "ذ": {"phoneme": "/ð/"}, "ر": {"phoneme": "/r/"}, "ز": {"phoneme": "/z/"}, "س": {"phoneme": "/s/"}, "ش": {"phoneme": "/ʃ/"}, "ص": {"phoneme": "/sˤ/"}, "ض": {"phoneme": "/dˤ/"}, "ط": {"phoneme": "/tˤ/"}, "ظ": {"phoneme": "/ðˤ/"}, "ع": {"phoneme": "/ʕ/"}, "غ": {"phoneme": "/ɣ/"}, "ف": {"phoneme": "/f/"}, "ق": {"phoneme": "/q/"}, "ك": {"phoneme": "/k/"}, "ل": {"phoneme": "/l/"}, "م": {"phoneme": "/m/"}, "ن": {"phoneme": "/n/"}, "ه": {"phoneme": "/h/"}, "و": {"phoneme": "/w/"}, "ي": {"phoneme": "/j/"}, "ة": {"phoneme": "/a/"}, "ا": {"phoneme": "/aː/"}}, "vowels": {"short": {"َ": "/a/", "ُ": "/u/", "ِ": "/i/"}, "long": {"ا": "/aː/", "و": "/uː/", "ي": "/iː/"}}, "diacritics": {"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled"}, "phonological_rules": {"al_prefix_moon": {"letters": "ا, ب, ج, ح, خ, ع, غ, ف, ق, ك, م, ه, و, ي", "ipa": "/al-/"}, "al_prefix_sun": {"letters": "تثدذرزسشصضطظلن", "ipa": "/a-/"}}, "lil_prefix": "/lal-/"}}}''')

def DIACRITIZE_HOOK(text, enable=False):
    """Recover short vowels BEFORE G2P. Tries CAMeL Tools, then Mishkal; identity fallback."""
    if not enable: return text
    try:
        from camel_tools.disambig.mle import MLEDisambiguator
        mle=MLEDisambiguator.pretrained(); out=[]
        for dd in mle.disambiguate(text.split()):
            diac=dd.analyses[0].analysis.get('diac') if dd.analyses else None
            out.append(diac or dd.word)
        return " ".join(out)
    except Exception: pass
    try:
        from mishkal.tashkeel import TashkeelClass
        return TashkeelClass().tashkeel(text)
    except Exception: return text

def get_rules(dialect):
    if not isinstance(dialect,str): return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]
    if "Egyptian" in dialect: return IPA_RULES["dialects"]["Egyptian"]
    if "Levantine" in dialect: return IPA_RULES["dialects"]["Levantine Arabic"]
    if "Khaliji" in dialect or "Gulf" in dialect: return IPA_RULES["dialects"]["Khaliji Arabic"]
    return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]

def _consume_diacritics(word,j,n,diacritics,units):
    shadda=False; vowel=None
    while j<n and word[j] in diacritics:
        if word[j]=='\u0651': shadda=True
        else:
            v=diacritics[word[j]]
            if v not in ("","doubled"): vowel=v.strip('/')
        j+=1
    if shadda and units: units.append(units[-1])
    if vowel: units.append(vowel)
    return j

def transcribe_arabic(word,letters,diacritics,sun_letters,rules):
    units=[]; i=0; n=len(word)
    while i<n:
        ch=word[i]
        if i==0 and ch=='\u0644' and i+1<n and word[i+1]=='\u0644':
            units.append(rules.get("lil_prefix","/lil-/").strip('/')); i+=2; continue
        if i==0 and ch=='\u0627' and i+1<n and word[i+1]=='\u0644':
            if i+2<n and word[i+2] in sun_letters:
                pre=rules["phonological_rules"]["al_prefix_sun"]["ipa"].strip('/')
                sp=letters.get(word[i+2],{}).get("phoneme","").split(" or ")[0].strip('/')
                units.append(f"{pre}{sp}{sp}"); i+=3
                i=_consume_diacritics(word,i,n,diacritics,units); continue
            else:
                units.append(rules["phonological_rules"]["al_prefix_moon"]["ipa"].strip('/')); i+=2; continue
        if ch=='\u0627':
            if i==0: units.append("ʔ")
            elif units and units[-1]=="a": units[-1]="aː"
            else: units.append("aː")
            i=_consume_diacritics(word,i+1,n,diacritics,units); continue
        # taa marbuta: /a/ in pausal, but DO NOT double if preceded by fatha (last unit "a")
        if ch == '\u0629':                                  # ة
            if not (units and units[-1] == "a"):
                units.append("a")
            i += 1; continue
        if ch in letters:
            ph=letters[ch]["phoneme"]
            if " or " in ph: ph=ph.split(" or ")[0]
            if ph: units.append(ph.strip('/'))
            i=_consume_diacritics(word,i+1,n,diacritics,units); continue
        i+=1
    joined="".join(units)
    return f"/{joined}/" if joined else ""

def transcribe_protected(text,dialect,diacritize=False):
    if text is None or text=="" : return ""
    text=DIACRITIZE_HOOK(str(text),enable=diacritize)
    rules=get_rules(dialect); letters=rules["letters"]
    diacritics={**rules["vowels"]["short"],**rules["diacritics"]}
    spec=rules["phonological_rules"]["al_prefix_sun"].get("letters")
    sun=set(spec.replace(",","").replace(" ","")) if spec else SUN_LETTERS
    out=[]
    for chunk in re.split(r'(<eng>.*?</eng>)',str(text)):
        if chunk.startswith("<eng>") and chunk.endswith("</eng>"): out.append(chunk)
        elif chunk.strip():
            for w in chunk.strip().split():
                p=transcribe_arabic(w,letters,diacritics,sun,rules)
                if p: out.append(p)
    return " ".join(out)


# ---- Short-vowel recovery via diacritize-then-G2P ----
# Without a diacritizer, output is a consonant+long-vowel skeleton.
print("skeleton :", transcribe_protected("كتب مدرس", "Egyptian", diacritize=False))
# With diacritization ON (requires: pip install camel-tools  OR  mishkal), short vowels appear.
# Pre-diacritized text also works directly:
print("diacritized:", transcribe_protected("كَتَبَ مُدَرِّس", "Modern Standard Arabic (MSA)", diacritize=False))
# To enable automatic diacritization, install a diacritizer and call with diacritize=True:
# !pip install camel-tools && camel_data -i disambig-mle-calima-msa-r13
# print(transcribe_protected("كتب مدرس", "Modern Standard Arabic (MSA)", diacritize=True))

# Self-test (run to validate the rules)

In [ ]:
# ---- Self-test: validates the rules without the full dataset ----
def _check(word, dialect, contains=None, equals=None, absent=None, diacritize=False):
    out = transcribe_protected(word, dialect, diacritize=diacritize); ok=True
    if equals is not None and out!=equals: ok=False
    if contains is not None and contains not in out: ok=False
    if absent is not None and absent in out: ok=False
    print(f"  [{'PASS' if ok else 'FAIL'}] {word} [{dialect[:4]}] -> {out}")
    return ok
CASES = [
    ("الشمس","Egyptian",dict(contains="ʃʃ")), ("الليل","Egyptian",dict(contains="ll")),
    ("الرحمن","Modern Standard Arabic (MSA)",dict(contains="rr")),
    ("القمر","Egyptian",dict(contains="el-")), ("الشمس","Khaliji Arabic",dict(contains="ɪ-")),
    ("سماء","Egyptian",dict(contains="ʔ",absent="æ")), ("آمن","Modern Standard Arabic (MSA)",dict(contains="ʔaː")),
    ("اسم","Egyptian",dict(contains="ʔ",absent="aːsm")), ("قال","Khaliji Arabic",dict(equals="/gaːl/")),
    ("جميل","Egyptian",dict(contains="ɡ")), ("جميل","Levantine Arabic",dict(contains="ʒ")),
    ("قلب","Khaliji Arabic",dict(contains="g")), ("قلب","Modern Standard Arabic (MSA)",dict(contains="q")),
    ("سكّر","Egyptian",dict(contains="kk")),
    ("كَتَبَ","Egyptian",dict(equals="/kataba/")), ("كِتَاب","Modern Standard Arabic (MSA)",dict(contains="kitaːb")),
    ("مُدَرِّس","Modern Standard Arabic (MSA)",dict(contains="mudarris")),
    ("مَدْرَسَة","Modern Standard Arabic (MSA)",dict(equals="/madrasa/",absent="aa")),
    ("شَجَرَة","Modern Standard Arabic (MSA)",dict(equals="/ʃadʒara/")),
    ("خرج ال<eng>boy</eng>","Egyptian",dict(contains="<eng>boy</eng>")),
]
passed=sum(_check(w,d,**kw) for w,d,kw in CASES)
print(f"\n{passed}/{len(CASES)} self-tests passed.")

# 3. Quick self-test (no dataset needed)

In [ ]:
for w,d in [("الشمس","Egyptian"),("الليل","Egyptian"),("الشمس","Khaliji Arabic"),
            ("قال","Khaliji Arabic"),("قال","Levantine Arabic"),("جبال","Modern Standard Arabic (MSA)"),
            ("إنت","Khaliji Arabic"),("خرج ال<eng>boy</eng> من الفصل","Egyptian")]:
    print(f"{w:<26} [{d:<28}] -> {transcribe_protected(w,d)}")

# 4. Batch-process `metadata.csv` files
*Recursively transcribes every `metadata.csv` (pipe-separated) under a folder, writing the `Phonemes` column. Mirrors the original tool's behavior, now with xlsx-synced rules.*

In [ ]:
def process_metadata_file(file_path):
    try:
        try:    df = pd.read_csv(file_path, sep='|', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, sep='|', encoding='windows-1256')
        cols = {c.lower().strip(): c for c in df.columns}
        t_col = cols.get('transcript_tagged') or cols.get('transcript')
        d_col = cols.get('main dialect') or cols.get('dialect')
        p_col = cols.get('phoneme') or cols.get('phonemes') or 'Phonemes'
        if not t_col or not d_col:
            print(f"  Skipping {file_path}: missing columns {list(df.columns)}"); return False
        df[p_col] = df.apply(lambda r: transcribe_protected(r[t_col], r[d_col]), axis=1)
        df.to_csv(file_path, index=False, sep='|', encoding='utf-8')
        print(f"  OK: {len(df)} rows transcribed."); return True
    except Exception as e:
        print(f"  Error {file_path}: {e}"); return False

main = input("Folder to scan for metadata.csv: ").strip()
if not os.path.exists(main):
    print("Path does not exist.")
else:
    n=0
    for root,_,files in os.walk(main):
        if "metadata.csv" in files:
            fp=os.path.join(root,"metadata.csv"); print("Found:",fp)
            if process_metadata_file(fp): n+=1
    print(f"\nDone. Processed {n} file(s).")